1. chargement des documents

In [20]:
import glob
from langchain_community.document_loaders import PyPDFLoader

# Load all PDF files from the specified directory
pdf_files = glob.glob("/home/soul/Desktop/FallahTech/DataRoom_FallahTech_PDF/*.pdf")
print(pdf_files)

all_pages = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    all_pages.extend(loader.load())


['/home/soul/Desktop/FallahTech/DataRoom_FallahTech_PDF/Index_DataRoom.pdf', '/home/soul/Desktop/FallahTech/DataRoom_FallahTech_PDF/Etude_Marche_Synthese.pdf', '/home/soul/Desktop/FallahTech/DataRoom_FallahTech_PDF/Registre_Personnel.pdf', '/home/soul/Desktop/FallahTech/DataRoom_FallahTech_PDF/Etats_Financiers_Historiques_NCT_2023_2025.pdf', '/home/soul/Desktop/FallahTech/DataRoom_FallahTech_PDF/Contrat_Cooperative_Type.pdf', '/home/soul/Desktop/FallahTech/DataRoom_FallahTech_PDF/Statuts_FallahTech.pdf', '/home/soul/Desktop/FallahTech/DataRoom_FallahTech_PDF/Etats_Financiers_Historiques_NCT.pdf']


2. découpage en chunks

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

chunks = splitter.split_documents(all_pages)


3. Embeddings

In [22]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model='mxbai-embed-large')

4. base vectorielle (FAISS or Chroma)

In [25]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="chroma_db")


In [ ]:
# test the vectorstore
print("Number of chunks:", len(chunks))
if len(chunks) > 0:
    print(chunks[0].page_content[:500])

Number of chunks: 68
INDEX DE LA DATA ROOM FALLAHTECH
Dossier d’Investissement - Série A Mise à jour : Mars     
 . Juridique
 . _Statuts_FallahTech.pdf : Statuts constitutifs de la société (SARL)
 . _Contrat_Cooperative_Type.pdf : Modèle de contrat de partenariat avec les
coopératives agricoles
 . Financier
 . _Etats_Financiers_Historiques_NCT_    _    .pdf : Bilans, comptes de
résultat et ﬂux de trésorerie historiques conformes aux Normes Comptables
Tunisiennes


5. retriever

In [26]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

6. LLM (LLAMA3.2)

In [27]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2", temperature=0)

7. chaine RAG (RetrievalQA)

In [31]:
from langchain_classic.chains import RetrievalQA

chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

8. Invocation

In [33]:
result = chain({"query": "combien d'articles dans contrat de partenariat strategique?"})
print(result['result'])
print("Source Documents:", result['source_documents'])

Je n'ai trouvé que 5 articles dans le contrat de partenariat stratégique.
Source Documents: [Document(id='ece509e5-102f-4ae6-81f7-d51fd34c7bf4', metadata={'producer': 'Skia/PDF m128', 'moddate': '2026-03-08T12:04:44+00:00', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/128.0.0.0 Safari/537.36', 'total_pages': 1, 'creationdate': '2026-03-08T12:04:44+00:00', 'page_label': '1', 'title': '1.2_Contrat_Cooperative_Type.md', 'page': 0, 'source': '/home/soul/Desktop/FallahTech/DataRoom_FallahTech_PDF/Contrat_Cooperative_Type.pdf'}, page_content='Faciliter la collecte des données agronomiques locales\nCentraliser la facturation et le paiement des abonnements\nArticle \x00 : Conditions ﬁnancières Le Partenaire bénéﬁcie d’une commission de \x00\x00%\nsur le chiﬀre d’aﬀaires généré par ses adhérents.\nArticle \x00 : Durée Le contrat est conclu pour une durée de \x00 ans, renouvelable par\ntacite reconduction.'), Document(id='4ccb5eb6-3c06-4487-ba

In [36]:
result = chain({"query": "Who is the CTO, and when was this person hired?"})
print(result['result'])

The CTO was hired in January.


In [ ]:
result = chain({"query": "How many employees work in the Tech department?"})
print(result['result'])